# Evaluate: face inpainting

Measure PSNR, SSIM and masked PSNR on the CelebA test images, for different mask sizes.

Run the cells from top to bottom. All settings are in the first code cell.

## 1. Settings

In [ ]:
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-inpainting"
LORA_DIR = "runs/faces/best"       # None = evaluate the base model without LoRA
TEST_LIST = "data/splits/faces_test.txt"
OUTPUT_DIR = "runs/faces/eval"

IMAGE_SIZE = 256
NUM_IMAGES = 50                    # for a quick test, use 2
MASK_LEVELS = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]    # part of the image that is hidden
STEPS = 50                         # sampling steps. For a quick test, use 10
GUIDANCE = 3.0

PROMPT = "a human face, photorealistic, natural features, clear, complete"
NEGATIVE_PROMPT = "blurry, distorted, incomplete, artifacts, cropped, bad quality, deformed"
SEED = 42

## 2. Imports and device

In [ ]:
import os
os.environ.setdefault("HF_HOME", os.path.abspath(".hf_cache"))   # model weights are kept in this folder

import math
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from peft import PeftModel
from PIL import Image, ImageDraw
from pytorch_msssim import ssim
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print("device:", DEVICE)

## 3. Pipeline

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

pipe = StableDiffusionInpaintPipeline.from_pretrained(BASE_MODEL, variant="fp16", torch_dtype=DTYPE,
                                                      safety_checker=None)

if LORA_DIR:
    pipe.unet = PeftModel.from_pretrained(pipe.unet, LORA_DIR).merge_and_unload()
    print("LoRA loaded from", LORA_DIR)
else:
    print("No LoRA: this is the base model")
pipe.to(DEVICE)
pipe.set_progress_bar_config(disable=True)

## 4. Masks and metrics

In [ ]:
def make_mask(size, percent, rng):
    """Square holes that cover `percent` of the image. The 5 % level uses 2 or 3 small squares."""
    count = rng.choice([2, 3]) if percent == 0.05 else 1
    side = int(math.sqrt(size * size * percent / count))
    mask = Image.new("L", (size, size), 0)
    draw = ImageDraw.Draw(mask)
    for _ in range(count):
        x = rng.randint(0, size - side)
        y = rng.randint(0, size - side)
        draw.rectangle([x, y, x + side, y + side], fill=255)
    return mask

def psnr(pred, gt):
    return (-10 * torch.log10(F.mse_loss(pred, gt) + 1e-8)).item()


def masked_psnr(pred, gt, mask):
    """PSNR of the pixels in the hole only."""
    m = mask.expand_as(gt)
    mse = ((pred - gt) ** 2 * m).sum() / m.sum()
    return (-10 * torch.log10(mse + 1e-8)).item()


def get_ssim(pred, gt):
    return ssim(pred[None], gt[None], data_range=1.0).item()

## 5. Evaluation

Each mask size runs two times: without the prompt and with the prompt.

In [ ]:
paths = [l.strip() for l in open(TEST_LIST) if l.strip()][:NUM_IMAGES]
os.makedirs(OUTPUT_DIR, exist_ok=True)
rows = []
examples = []        # some images to show at the end

for mode in ["off", "on"]:                      # without the prompt, then with the prompt
    prompt = PROMPT if mode == "on" else ""
    negative = NEGATIVE_PROMPT if mode == "on" else None
    for percent in MASK_LEVELS:
        scores = []
        for i, path in enumerate(tqdm(paths, desc=f"prompt {mode}, mask {int(percent * 100)}%")):
            img = Image.open(path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
            mask = make_mask(IMAGE_SIZE, percent, random.Random(SEED * 1000 + i))   # same mask for off and on
            masked = Image.composite(Image.new("RGB", img.size, 0), img, mask)
            extra = {}
            out = pipe(prompt=prompt, negative_prompt=negative, image=img, mask_image=mask,
                       height=IMAGE_SIZE, width=IMAGE_SIZE, num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                       generator=torch.Generator("cpu").manual_seed(SEED), **extra).images[0]

            gt, pred, m = to_tensor(img), to_tensor(out), to_tensor(mask)
            scores.append((psnr(pred, gt), get_ssim(pred, gt), masked_psnr(pred, gt, m)))
            if i == 0:
                examples.append((f"prompt {mode}, mask {int(percent * 100)}%", [img, mask, masked, out]))

        p, s, mp = np.mean(scores, axis=0)
        rows.append({"prompt": mode, "mask_%": int(percent * 100), "PSNR": p, "SSIM": s, "masked_PSNR": mp})
        print(f"prompt {mode} | mask {int(percent * 100):>2}% | PSNR {p:.2f} | SSIM {s:.3f} | masked PSNR {mp:.2f}")

results = pd.DataFrame(rows)
results.to_csv(f"{OUTPUT_DIR}/results.csv", index=False)
results.round(3)

## 6. Examples

In [ ]:
names = ["original", "mask", "masked", "output"]
for title, tiles in examples:
    fig, axes = plt.subplots(1, len(tiles), figsize=(3 * len(tiles), 3.3))
    for ax, tile, name in zip(axes, tiles, names):
        ax.imshow(tile, cmap="gray")
        ax.set_title(name)
        ax.axis("off")
    fig.suptitle(title)
    fig.savefig(f"{OUTPUT_DIR}/{title.replace(', ', '_').replace(' ', '_').replace('%', '')}.png")
    plt.show()